In [0]:
!pip install haversine

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
from haversine import haversine
from pyspark.sql.types import FloatType

def dist_haversine(lat_o, lon_o, lat_d, lon_d):
    return haversine((lat_o, lon_o), (lat_d, lon_d))

spark.udf.register("dist_haversine", dist_haversine, FloatType())

<function __main__.dist_haversine(lat_o, lon_o, lat_d, lon_d)>

In [0]:

spark.sql("""
    WITH tbl_features AS (
    SELECT 
        day(to_timestamp(datahora_entregue, 'dd/MM/yyy HH:mm')) as dia,
        dayofweek(to_timestamp(datahora_entregue, 'dd/MM/yyy HH:mm')) as dia_semana,
        month(to_timestamp(datahora_entregue, 'dd/MM/yyy HH:mm')) as mes,
        hour(to_timestamp(datahora_entregue, 'dd/MM/yyy HH:mm')) as hora,
        duracao,
        distancia,
        vlr_pago,
        latitude_origem,
        longitude_origem,
        latitude_destino,
        longitude_destino
    FROM sandbox.prcg.transacoes
    )
    SELECT 
    dia,
    dia_semana,
    mes,
    hora,
    case when hora < 12 and hora > 6 then 1
        when hora >= 12 and hora <= 18 then 2
        else 3 end as periodo,
    duracao,
    vlr_pago,
    latitude_origem,
    longitude_origem,
    latitude_destino,
    longitude_destino,
    dist_haversine(latitude_origem, longitude_origem, latitude_destino, longitude_destino) as distancia
    FROM tbl_features
""").createOrReplaceTempView("tbl_features")

In [0]:
spark.sql(
    """
    SELECT 
        vlr_pago,
        dia,
        dia_semana,
        mes,
        hora,
        periodo,
        distancia,
        latitude_origem,
        longitude_origem,
        latitude_destino,
        longitude_destino
    FROM tbl_features
"""
).createOrReplaceTempView("tbl_ml")

In [0]:
spark.sql(
    """
    SELECT 
    vlr_pago,
    dia,
    dia_semana,
    mes,
    hora,
    periodo,
    distancia,
    duracao
    FROM tbl_features
"""
).createOrReplaceTempView("tbl_eda")